In [0]:
from olist_lakehouse.io import read_managed_table, write_managed_table
from pyspark.sql import Window
from pyspark.sql.functions import (
    avg,
    col,
    countDistinct,
    rank,
)
from pyspark.sql.functions import (
    count as _count,
)
from pyspark.sql.functions import (
    sum as _sum,
)

In [0]:
dbutils.widgets.text("catalog", "")

In [0]:
catalog = dbutils.widgets.get("catalog")

## Sales by category table
Which categories generate more revenue? Which one has the biggest average order ticket?

**Table requirements:** 

- Grouping by category (in english)
- Must be ordered by `revenue_rank`
- Only orders with `order_status` = `'delivered'`

| Metric             | Description                                              |
| -------------------| ---------------------------------------------------------|
| `total_revenue`    | Sum of `price` of all sold items                         |
| `total_orders`     | Distinct orders containing the item from the category    |
| `total_items_sold` | Total items sold                                         |
| `avg_order_ticket` | Average order revenue (`total_revenue` / `total_orders`) |
| `avg_item_price`   | Average item price                                       |
| `revenue_rank`     | Revenue rank category                                    |

In [0]:
orders = read_managed_table(catalog, "silver", "orders")
order_items = read_managed_table(catalog, "silver", "order_items")
products = read_managed_table(catalog, "silver", "products")

In [0]:
orders_and_products = order_items.join(products, on="product_id", how="left").join(orders, on="order_id", how="left")
orders_and_products_filtered = orders_and_products.filter("order_status = 'delivered'")

In [0]:
sales_by_category = orders_and_products_filtered.groupBy("product_category_name_english").agg(
    _sum("price").alias("total_revenue"),
    countDistinct("order_id").alias("total_orders"),
    _count("*").alias("total_items_sold"),
    (_sum("price") / countDistinct("order_id")).alias("avg_order_ticket"),
    avg("price").alias("avg_item_price"),
)

In [0]:
wn_revenue = Window.orderBy(col("total_revenue").desc(), col("avg_order_ticket").desc())

sales_by_category_ranked = sales_by_category.withColumn("revenue_rank", rank().over(wn_revenue))

In [0]:
write_managed_table(
    sales_by_category_ranked,
    mode="overwrite",
    table_full_path=f"{catalog}.gold.sales_by_category",
    options={"overwriteSchema": "true"}
)